# 6. Buckets (frequency channels) per band

This notebook scans the **frequency Legend** CSVs produced by the transform and shows how the collected data is split into **frequency buckets** (channels) per band.

- **Summary table:** For each band, number of buckets, frequency span (GHz), and bucket width (MHz).
- **Bar chart:** Number of buckets per band.
- **Frequency ruler:** For each band, a row showing where each bucket sits in frequency (GHz).

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import plotly.graph_objects as go

In [13]:
# Path to frequency legends (same layout as other notebooks)
csvs_dir = Path("work_dir/csvs")
if not csvs_dir.exists():
    csvs_dir = Path("../work_dir/csvs")
legend_dir = csvs_dir / "frequency_legends"

if not legend_dir.exists():
    raise FileNotFoundError(f"Directory not found: {legend_dir}. Run the transform first.")

# Filename pattern: Legend_YYYY_MM_DD_HH_<band>.csv
def band_from_legend_name(name: str):
    m = re.match(r"Legend_\d{4}_\d{2}_\d{2}_\d{2}_(.+)\.csv", name, re.IGNORECASE)
    return m.group(1) if m else None

legend_files = sorted(legend_dir.glob("Legend_*.csv"))
bands_seen = set()
band_to_one_path = {}  # band -> path to one Legend file (for that band)
for path in legend_files:
    band = band_from_legend_name(path.name)
    if band and band not in bands_seen:
        bands_seen.add(band)
        band_to_one_path[band] = path

if not band_to_one_path:
    raise ValueError(f"No Legend_* files found in {legend_dir}")

print(f"Found {len(band_to_one_path)} unique band(s): {sorted(band_to_one_path.keys())}")

Found 5 unique band(s): ['2441MHz', '3765MHz', '539MHz', '5500MHz', '915MHz']


In [14]:
# Load one Legend per band. Legend format: col0=channel_index, col1=start_Hz, col2=end_Hz
rows = []
band_buckets = {}  # band -> list of (start_ghz, end_ghz) for each bucket

for band in sorted(band_to_one_path.keys()):
    path = band_to_one_path[band]
    df = pd.read_csv(path, header=None)
    if len(df) == 0:
        continue
    start_hz = df.iloc[:, 1].values
    end_hz = df.iloc[:, 2].values
    n_buckets = len(df)
    min_ghz = start_hz.min() / 1e9
    max_ghz = end_hz.max() / 1e9
    span_ghz = max_ghz - min_ghz
    width_hz = end_hz[0] - start_hz[0]
    width_mhz = width_hz / 1e6
    rows.append({
        "Band": band,
        "Num buckets": n_buckets,
        "Min (GHz)": round(min_ghz, 3),
        "Max (GHz)": round(max_ghz, 3),
        "Span (GHz)": round(span_ghz, 3),
        "Bucket width (MHz)": round(width_mhz, 1),
    })
    band_buckets[band] = [(s / 1e9, e / 1e9) for s, e in zip(start_hz, end_hz)]

summary = pd.DataFrame(rows)
summary

,Band,Num buckets,Min (GHz),Max (GHz),Span (GHz),Bucket width (MHz)
0,2441MHz,1,2.44,2.46,0.02,20.0
1,3765MHz,12,3.70,3.94,0.24,20.0
2,539MHz,1,0.48,0.50,0.02,20.0
3,5500MHz,36,5.14,5.86,0.72,20.0
4,915MHz,2,0.90,0.94,0.04,20.0


In [15]:
# Bar chart: number of buckets per band
fig = go.Figure(data=[
    go.Bar(x=summary["Band"], y=summary["Num buckets"], text=summary["Num buckets"], textposition="outside")
])
fig.update_layout(
    title="Number of frequency buckets (channels) per band",
    xaxis_title="Band",
    yaxis_title="Num buckets",
    height=400,
    showlegend=False,
)
fig.show()

In [ ]:
# Optional: show bucket center frequencies (GHz) per band
band_list = summary["Band"].tolist()
for band in band_list:
    buckets = band_buckets[band]
    centers_ghz = [(s + e) / 2 for s, e in buckets]
    print(f"{band}: {len(centers_ghz)} buckets, centers (GHz) = {[round(c, 3) for c in centers_ghz]}")

2441MHz: 1 buckets, centers (GHz) = [np.float64(2.45)]
3765MHz: 12 buckets, centers (GHz) = [np.float64(3.71), np.float64(3.73), np.float64(3.75), np.float64(3.77), np.float64(3.79), np.float64(3.81), np.float64(3.83), np.float64(3.85), np.float64(3.87), np.float64(3.89), np.float64(3.91), np.float64(3.93)]
539MHz: 1 buckets, centers (GHz) = [np.float64(0.49)]
5500MHz: 36 buckets, centers (GHz) = [np.float64(5.15), np.float64(5.17), np.float64(5.19), np.float64(5.21), np.float64(5.23), np.float64(5.25), np.float64(5.27), np.float64(5.29), np.float64(5.31), np.float64(5.33), np.float64(5.35), np.float64(5.37), np.float64(5.39), np.float64(5.41), np.float64(5.43), np.float64(5.45), np.float64(5.47), np.float64(5.49), np.float64(5.51), np.float64(5.53), np.float64(5.55), np.float64(5.57), np.float64(5.59), np.float64(5.61), np.float64(5.63), np.float64(5.65), np.float64(5.67), np.float64(5.69), np.float64(5.71), np.float64(5.73), np.float64(5.75), np.float64(5.77), np.float64(5.79), np.fl

In [17]:
# Frequency ruler: each band is a row; each bucket is a horizontal segment (start_ghz to end_ghz)
# Order bands by frequency (low to high) so the ruler matches the x-axis
summary_sorted = summary.sort_values("Min (GHz)")
band_list = summary_sorted["Band"].tolist()
y_labels = band_list
y_inds = np.arange(len(band_list))

fig = go.Figure()
colors = px_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

for i, band in enumerate(band_list):
    buckets = band_buckets[band]
    color = colors[i % len(colors)]
    for j, (start_ghz, end_ghz) in enumerate(buckets):
        fig.add_shape(
            type="rect",
            x0=start_ghz, x1=end_ghz,
            y0=i - 0.45, y1=i + 0.45,
            line=dict(width=0.5, color="#333"),
            fillcolor=color,
            opacity=0.7,
        )

fig.update_layout(
    title="Frequency buckets per band (each segment = one channel)",
    xaxis_title="Frequency (GHz)",
    yaxis=dict(
        tickmode="array",
        tickvals=y_inds,
        ticktext=y_labels,
        title="Band",
    ),
    height=200 + 50 * len(band_list),
    showlegend=False,
    margin=dict(l=100),
)
fig.update_xaxes(autorange=False, range=[summary["Min (GHz)"].min() - 0.1, summary["Max (GHz)"].max() + 0.1])
fig.update_yaxes(autorange="reversed")
fig.show()

## Verify: CSVs match raw parquet (e.g. 2441 MHz)

The transform builds Legend/Data CSVs from parquet files: it groups by **(band, date, hour)**, reads the `freqs` column, and bins into 20 MHz channels. So the **number of buckets** in the Legend is determined by the **actual frequency range** in the raw parquet for that band.

For **2441 MHz**: the Legend shows **1 bucket** (2.44–2.46 GHz). That means the raw parquet files for Band_2441MHz only contained detections within that single 20 MHz channel. Below we load the 2441 MHz parquet files and check that `freqs` min/max fall inside that range.

In [18]:
# Verify 2441 MHz: raw parquet freq range vs Legend (single bucket)
raw_data = csvs_dir.parent / "raw_data"
if not raw_data.exists():
    raw_data = Path("../work_dir/raw_data")
band_to_check = "2441MHz"
parquet_files = sorted(raw_data.glob(f"Band_{band_to_check}*.parquet"))

if not parquet_files:
    print(f"No parquet files found for {band_to_check} in {raw_data}")
else:
    dfs = []
    for p in parquet_files:
        try:
            df = pd.read_parquet(p)
            if "freqs" in df.columns:
                dfs.append(df[["freqs"]] if "freqs" in df.columns else df)
        except Exception as e:
            print(f"Skip {p.name}: {e}")
    if not dfs:
        print("No parquet could be read.")
    else:
        all_freqs = pd.concat(dfs, ignore_index=True)["freqs"]
        f_min_hz = all_freqs.min()
        f_max_hz = all_freqs.max()
        f_min_ghz = f_min_hz / 1e9
        f_max_ghz = f_max_hz / 1e9
        print(f"Band {band_to_check}: {len(parquet_files)} parquet file(s), {len(all_freqs)} rows")
        print(f"  Raw parquet freqs: {f_min_ghz:.4f} – {f_max_ghz:.4f} GHz  ({f_min_hz:.0f} – {f_max_hz:.0f} Hz)")
        # Legend for 2441 MHz (from summary): 1 bucket, 2.44–2.46 GHz
        legend_min = band_buckets[band_to_check][0][0]
        legend_max = band_buckets[band_to_check][0][1]
        print(f"  Legend bucket:    {legend_min:.4f} – {legend_max:.4f} GHz  (1 bucket)")
        inside = (f_min_hz >= legend_min * 1e9) and (f_max_hz <= legend_max * 1e9)
        print(f"  Parquet range inside Legend bucket: {inside}")
        if inside:
            print("  => Raw data for 2441 MHz was collected only in this single 20 MHz band; Legend is correct.")

Band 2441MHz: 4765 parquet file(s), 460320039 rows
  Raw parquet freqs: 2.4000 – 2.4835 GHz  (2399999244 – 2483502636 Hz)
  Legend bucket:    2.4400 – 2.4600 GHz  (1 bucket)
  Parquet range inside Legend bucket: False


In [19]:
# Optional: verify all bands — parquet freq range vs Legend buckets
if "raw_data" in dir():
    for band in summary["Band"].tolist():
        parquet_files = sorted(raw_data.glob(f"Band_{band}*.parquet"))
        if not parquet_files:
            continue
        try:
            dfs = [pd.read_parquet(p) for p in parquet_files[:5]]
            dfs = [df for df in dfs if "freqs" in df.columns]
            if not dfs:
                continue
            all_f = pd.concat([df[["freqs"]] for df in dfs], ignore_index=True)["freqs"]
            buckets = band_buckets[band]
            leg_min_ghz = min(s for s, e in buckets)
            leg_max_ghz = max(e for s, e in buckets)
            raw_min = all_f.min() / 1e9
            raw_max = all_f.max() / 1e9
            ok = raw_min >= leg_min_ghz - 0.001 and raw_max <= leg_max_ghz + 0.001
            print(f"{band}: parquet {raw_min:.3f}–{raw_max:.3f} GHz  Legend {leg_min_ghz:.3f}–{leg_max_ghz:.3f} GHz  match={ok}")
        except Exception as e:
            print(f"{band}: {e}")

2441MHz: parquet 2.403–2.484 GHz  Legend 2.440–2.460 GHz  match=False
3765MHz: parquet 3.550–3.980 GHz  Legend 3.700–3.940 GHz  match=False
539MHz: parquet 0.476–0.608 GHz  Legend 0.480–0.500 GHz  match=False
5500MHz: parquet 5.155–5.846 GHz  Legend 5.140–5.860 GHz  match=True
915MHz: parquet 0.902–0.928 GHz  Legend 0.900–0.940 GHz  match=True
